# T-Seeded Model: Step-by-Step Example

This notebook demonstrates the T-seeded ODE model using repository functions.

**Purpose**: Understand the time-dependent fuel cycle computation for a single parameter combination.

**Model**: ODE-based model tracking tritium through plasma, fuel cycle systems (IFC, OFC, storage).

## Step 1: Import Required Functions and Constants

Import all necessary functions and constants from the repository.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Add repository root to path
_cwd = Path.cwd().resolve()
_candidates = [_cwd, *_cwd.parents]
_repo_root = next((p for p in _candidates if (p / "src").is_dir()), None)
if _repo_root is not None and str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

# Import physical constants from registry
from src.registry.parameter_registry import (
    lambda_T,
    tritium_mass,
    REACTION_ENERGY_BY_CHANNEL
)

# Import reactivity functions
from src.physics.reactivity_functions import (
    sigmav_DT_BoschHale,
    sigmav_DD_BoschHale
)

# Import T-seeded solver
from src.physics.Tseeded_functions import solve_ode_system

print("Repository Functions and Constants Loaded:")
print(f"  λ_T (tritium decay constant) = {lambda_T:.6e} s⁻¹")
print(f"  Tritium mass = {tritium_mass:.6e} kg")
print()

## Step 2: Define Input Parameters

Set the plasma, breeding, and fuel cycle parameters for this example case.

In [ ]:
# Plasma operational parameters
V_plasma = 2000.0               # Plasma volume (m³)
T_i = 20.0                      # Ion temperature (keV)
n_tot = 7.0e19                 # Total particle density (m⁻³)
tau_p_T = 5.0                   # Tritium confinement time (s)

# Breeding ratios
TBR_DT = 1.3                    # D-T tritium breeding ratio
TBR_DDn = 1.0                   # DD neutron tritium breeding ratio

# Fuel cycle parameters
tau_ifc = 4.0 * 3600            # Inner fuel cycle time constant (s) - 4 hours
tau_ofc = 2.0 * 3600            # Outer fuel cycle time constant (s) - 2 hours
N_ofc_0 = 0.0                   # Initial tritium in outer fuel cycle (atoms)
N_ifc_0 = 0.0                   # Initial tritium in inner fuel cycle (atoms)
N_stor_0 = 0.0                    # Initial tritium in storage (atoms)

# Target and initial conditions
I_target = 0.5                  # Target tritium inventory (kg)
N_target = I_target / tritium_mass  # Target in atoms
T_frac_0 = 1e-6                 # Initial tritium fraction in plasma

# Simulation parameters
t_end = 1e9                     # Maximum simulation time (s)

print("Input Parameters:")
print(f"  V_plasma = {V_plasma} m³")
print(f"  T_i = {T_i} keV")
print(f"  n_tot = {n_tot:.2e} m⁻³")
print(f"  τ_p_T = {tau_p_T} s")
print(f"  TBR_DT = {TBR_DT}")
print(f"  TBR_DDn = {TBR_DDn}")
print(f"  τ_ifc = {tau_ifc/3600:.1f} hours")
print(f"  τ_ofc = {tau_ofc/3600:.1f} hours")
print(f"  I_target = {I_target} kg ({N_target:.6e} atoms)")
print(f"  Initial T fraction = {T_frac_0:.2e}")
print()

## Step 3: Calculate Fusion Reactivities

Compute reaction rate coefficients at the operating temperature.

In [ ]:
# Calculate reactivities at T_i
sigmav_DT = sigmav_DT_BoschHale(T_i)
sigmav_DD_tot, sigmav_DD_n, sigmav_DD_p = sigmav_DD_BoschHale(T_i)

print(f"Reaction Rate Coefficients at T_i = {T_i} keV:")
print(f"  ⟨σv⟩_DD_tot = {sigmav_DD_tot:.6e} m³/s")
print(f"  ⟨σv⟩_DD_p (DD→T+p) = {sigmav_DD_p:.6e} m³/s")
print(f"  ⟨σv⟩_DD_n (DD→³He+n) = {sigmav_DD_n:.6e} m³/s")
print(f"  ⟨σv⟩_DT = {sigmav_DT:.6e} m³/s")
print()

## Step 4: Solve the T-Seeded ODE System

Run the ODE solver to track tritium evolution through all fuel cycle systems.

In [ ]:
# Call the T-seeded solver
result = solve_ode_system(
    V_plasma=V_plasma,
    n_tot=n_tot,
    tau_p_T=tau_p_T,
    tau_ifc=tau_ifc,
    tau_ofc=tau_ofc,
    TBR_DT=TBR_DT,
    TBR_DDn=TBR_DDn,
    N_target=N_target,
    sigmav_DD_p=sigmav_DD_p,
    sigmav_DD_n=sigmav_DD_n,
    sigmav_DT=sigmav_DT,
    N_ofc_0=N_ofc_0,
    N_ifc_0=N_ifc_0,
    N_stor_0=N_stor_0,
    T_frac_0=T_frac_0,
    t_end=t_end
)

print("T-Seeded ODE Model Results:")
print(f"  Solution success: {result['sol_success']}")
if result['sol_success']:
    print(f"  t_startup = {result['t_startup']:.2f} s ({result['t_startup']/3600:.2f} hours, {result['t_startup']/86400:.2f} days)")
    print(f"  Final n_T = {result['n_T']:.6e} m⁻³")
    print(f"  Final n_D = {result['n_D']:.6e} m⁻³")
    print(f"  Final N_ifc = {result['N_ifc']:.6e} atoms")
    print(f"  Final N_ofc = {result['N_ofc']:.6e} atoms")
    print(f"  Final N_stor = {result['N_stor']:.6e} atoms")
    print(f"  Total T inventory = {(result['n_T']*V_plasma + result['N_ifc'] + result['N_ofc'] + result['N_stor'])*tritium_mass:.3f} kg")
else:
    print(f"  Error: {result['error']}")
print()

## Step 5: Extract and Plot Time Evolution

Analyze the complete time evolution of tritium through all systems.

In [ ]:
if result['sol_success'] and 'sol' in result:
    # Extract time evolution
    sol = result['sol']
    t_eval = sol.t
    
    # State variables: [T_frac, N_ifc, N_ofc, N_stor]
    T_frac = sol.y[0]
    N_ifc = sol.y[1]
    N_ofc = sol.y[2]
    N_stor = sol.y[3]
    
    # Derive other quantities
    n_T = T_frac * n_tot
    n_D = (1 - T_frac) * n_tot
    N_plasma = n_T * V_plasma
    
    # Total inventory (kg)
    I_total = (N_plasma + N_ifc + N_ofc + N_stor) * tritium_mass
    
    # Plot evolution
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: Densities in plasma
    ax = axes[0, 0]
    ax.plot(t_eval/3600, n_T, 'r-', label='n_T (tritium)', linewidth=2)
    ax.plot(t_eval/3600, n_D, 'b-', label='n_D (deuterium)', linewidth=2)
    ax.set_xlabel('Time (hours)')
    ax.set_ylabel('Density (m⁻³)')
    ax.set_title('Plasma Densities Evolution')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_yscale('log')
    
    # Plot 2: Tritium in fuel cycle systems
    ax = axes[0, 1]
    ax.plot(t_eval/3600, N_plasma*tritium_mass, label='Plasma', linewidth=2)
    ax.plot(t_eval/3600, N_ifc*tritium_mass, label='IFC (inner)', linewidth=2)
    ax.plot(t_eval/3600, N_ofc*tritium_mass, label='OFC (outer)', linewidth=2)
    ax.plot(t_eval/3600, N_stor*tritium_mass, label='Storage', linewidth=2)
    ax.axhline(y=I_target, color='k', linestyle='--', label=f'Target ({I_target} kg)')
    ax.set_xlabel('Time (hours)')
    ax.set_ylabel('Tritium Inventory (kg)')
    ax.set_title('Tritium Distribution Across Systems')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 3: Total inventory
    ax = axes[1, 0]
    ax.plot(t_eval/3600, I_total, 'g-', linewidth=2)
    ax.axhline(y=I_target, color='r', linestyle='--', label=f'Target ({I_target} kg)')
    if result['t_startup'] > 0:
        ax.axvline(x=result['t_startup']/3600, color='orange', linestyle='--', 
                   label=f'Startup ({result["t_startup"]/3600:.1f} h)')
    ax.set_xlabel('Time (hours)')
    ax.set_ylabel('Total Tritium Inventory (kg)')
    ax.set_title('Total Tritium Inventory Evolution')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 4: Tritium fraction in plasma
    ax = axes[1, 1]
    ax.plot(t_eval/3600, T_frac, 'purple', linewidth=2)
    ax.set_xlabel('Time (hours)')
    ax.set_ylabel('Tritium Fraction')
    ax.set_title('Tritium Fraction in Plasma')
    ax.grid(True, alpha=0.3)
    ax.set_yscale('log')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nKey Result: Startup time = {result['t_startup']/3600:.2f} hours ({result['t_startup']/86400:.2f} days)")

## Step 6: Calculate Fusion Powers at Steady State

Compute the fusion power at the final state.

In [ ]:
if result['sol_success']:
    n_T_final = result['n_T']
    n_D_final = result['n_D']
    
    # Get reaction energies
    E_DD_n = REACTION_ENERGY_BY_CHANNEL['sigmav_DD_n']
    E_DD_p = REACTION_ENERGY_BY_CHANNEL['sigmav_DD_p']
    E_DT = REACTION_ENERGY_BY_CHANNEL['sigmav_DT']
    
    # Calculate reaction rates (reactions/s)
    R_DD_n = 0.5 * n_D_final * n_D_final * sigmav_DD_n * V_plasma
    R_DD_p = 0.5 * n_D_final * n_D_final * sigmav_DD_p * V_plasma
    R_DT = n_D_final * n_T_final * sigmav_DT * V_plasma
    
    # Calculate powers (W)
    P_DD_n = R_DD_n * E_DD_n
    P_DD_p = R_DD_p * E_DD_p
    P_DT = R_DT * E_DT
    P_total = P_DD_n + P_DD_p + P_DT
    
    print("Fusion Powers at Steady State:")
    print(f"  P_DD_n (DD→³He+n) = {P_DD_n/1e6:.2f} MW")
    print(f"  P_DD_p (DD→T+p) = {P_DD_p/1e6:.2f} MW")
    print(f"  P_DT (DT→⁴He+n) = {P_DT/1e6:.2f} MW")
    print(f"  P_total = {P_total/1e6:.2f} MW")
    print()
    
    # Tritium balance at steady state
    Tdot_DDp = R_DD_p
    Tdot_DT_bred = TBR_DT * R_DT
    Tdot_DDn_bred = TBR_DDn * R_DD_n
    Tdot_production = Tdot_DDp + Tdot_DT_bred + Tdot_DDn_bred
    
    Tdot_burn = R_DT
    N_total = n_T_final * V_plasma + result['N_ifc'] + result['N_ofc'] + result['N_stor']
    Tdot_decay = lambda_T * N_total
    Tdot_confinement = n_T_final * V_plasma / tau_p_T
    Tdot_loss = Tdot_burn + Tdot_decay + Tdot_confinement
    
    print("Tritium Balance at Steady State (atoms/s):")
    print(f"  Production (DDp) = {Tdot_DDp:.6e}")
    print(f"  Production (DT bred) = {Tdot_DT_bred:.6e}")
    print(f"  Production (DDn bred) = {Tdot_DDn_bred:.6e}")
    print(f"  Total production = {Tdot_production:.6e}")
    print(f"  Burn (DT) = {Tdot_burn:.6e}")
    print(f"  Decay = {Tdot_decay:.6e}")
    print(f"  Confinement loss = {Tdot_confinement:.6e}")
    print(f"  Total loss = {Tdot_loss:.6e}")
    print(f"  Net balance = {Tdot_production - Tdot_loss:.6e}")

## Summary

This example demonstrated:
1. Importing constants and functions from the repository
2. Setting up input parameters for the T-seeded ODE model
3. Computing fusion reactivities at a specific temperature
4. Solving the ODE system for time-dependent tritium evolution
5. Visualizing tritium distribution across plasma and fuel cycle systems
6. Calculating fusion powers and tritium balance at steady state

The T-seeded model provides detailed time evolution of tritium through all fuel cycle systems, offering more physical realism than the lump model at the cost of computational time.